<a href="https://colab.research.google.com/github/hig3r/mva/blob/2026/E02/mva_2026_E02_02_multiple_linear_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mva_2026_E02_02_multiple_linear_regression.ipynb 線形重回帰モデル with scikit-learn
- 学籍番号 [ ]
- 氏名 [ ]

課題2の解答をここに記入して．
- 切片 $\beta_0=$
- weight の係数 $\beta_1=$
- transparency の係数 $\beta_2=$

In [ ]:
import numpy as np # 数値計算ライブラリ
import pandas as pd # データ解析ライブラリ
import matplotlib.pyplot as plt # グラフ描画ライブラリ
%matplotlib inline

from sklearn.linear_model import LinearRegression # scikit-learn の線形重回帰ライブラリ

!pip install japanize_matplotlib
import japanize_matplotlib # 日本語フォント

from mpl_toolkits.mplot3d import Axes3D # 3Dプロット用のインポート
import plotly.graph_objects as go # 3次元インタラクティブプロット

## 例：成績データ

### データの取得と整形

In [ ]:
#   「Pythonで理解する統計解析の基礎」 谷合廣紀，辻 真吾，技術評論社，2018. のデータを，dataframe形式で読み込み
df=pd.read_csv("https://github.com/ghmagazine/python_stat_sample/raw/master/data/ch12_scores_reg.csv")
df

- 統計モデルとして，線形重回帰 LinearRegression を採用
- Pythonライブラリとして，sklearn=scikit-learnを採用
  - 使い方や，結果の意味は全てここに書いてある
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

In [ ]:
model=LinearRegression() # 上でfrom sklearn.linear_model import LinearRegression しているのでこれで使える

期末テストを目的変数$y$, 小テストを説明変数$x_1$, 睡眠時間を説明変数$x_2$ とする．行列Xとベクトルyを作る．

sklearn のLinearRegressionは，行列$X$に関して，1の列の追加や，中心化を自動的にやってくれるので，このdf相当のデータ
$$
  \begin{pmatrix}
    {x}_{11}&\quad {x}_{{1}{2}}\\
    \vdots & \vdots \\
    \vdots & \vdots \\    
    {x}_{{n}{1}}&\quad {x}_{{n}{2}}\\
  \end{pmatrix}
$$

をそのまま渡せばよい．

授業でやったこの形に整形しなくてよい.
$$
  X=
  \begin{pmatrix}
    1 & {x}_{11}-\overline{x}_{{\bullet}{1}}&\quad {x}_{{1}{2}}-{\overline{x}}_{{\bullet}{2}}\\
    \vdots & \vdots \\
    \vdots & \vdots \\    
    1 & {x}_{{n}{1}}-\overline{x}_{{\bullet}{1}}&\quad {x}_{{n}{2}}-\overline{x}_{{\bullet}{2}}\\
  \end{pmatrix}
$$


In [ ]:
X_train=df.drop(["期末テスト", "通学方法"],axis=1) # 列を削除した dataframe
X_train_np = X_train.to_numpy() # NumPyの行列=2次元配列に変換
X_train_np

yはdfから1列だけ取り出せばよい．これは，dfのSeriesと呼ばれる形．NumPy配列に変換すると1次元配列になる

In [ ]:
y_train=df["期末テスト"] # 1列だけ取り出した Series
y_train_np = y_train.to_numpy() # numpyの1次元配列に変換

実は，`.to_numpy()`しなくても，`LinearRegression`が自動的に変換してくれるのだが，ここではやっておいた．

_train は，母数の推定に使う学習データ，訓練データ，教師データのこと．

$X$は特徴量，$y$はターゲットと呼ばれたりもする．

In [ ]:
# 変換後のデータ型と形状（shape）を確認します
print("X_train_np の型:", type(X_train_np))
print("X_train_np の shape:", X_train_np.shape)
print("y_train_np の型:", type(y_train_np))
print("y_train_np の shape:", y_train_np.shape)

### 母数の推定：偏回帰係数と切片

In [ ]:
# モデルの学習
model.fit(X_train_np, y_train_np) # 固定順序の引数
# 結果は，返り値でなく，modelのpropertyとして保存される．

In [ ]:
# 偏回帰係数と切片、決定係数を取得して表示
print("説明変数:", X_train.columns.tolist())
print("偏回帰係数β1,β2:", model.coef_)
print("切片β0:", model.intercept_)
print("決定係数 (R^2):", model.score(X_train, y_train))

### 予測

xのデータを与えて，その値に対してyを計算させる．

In [ ]:
model.predict(np.array([[2,3]]))

そもそも母数の推定に使ったxのデータに対して予測させる

In [ ]:
model.predict(X_train_np)

### 可視化
このセクションの中身は理解せず実行すればよい

In [ ]:
# データの準備
x1 = X_train_np[:, 0] # 小テスト
x2 = X_train_np[:, 1] # 睡眠時間
y = y_train_np        # 期末テスト

# 予測値の計算
y_pred = model.predict(X_train_np)

# 回帰平面を描画するためのグリッドデータを作成
x1_range = np.linspace(x1.min() - 0.5, x1.max() + 0.5, 20)
x2_range = np.linspace(x2.min() - 0.5, x2.max() + 0.5, 20)
X1_grid, X2_grid = np.meshgrid(x1_range, x2_range)

# 各グリッドポイントでの予測値を計算 (Z軸の値)
# 予測式: y = b0 + b1*x1 + b2*x2
Y_grid = model.intercept_ + model.coef_[0] * X1_grid + model.coef_[1] * X2_grid

# グラフの描画設定
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 実際の学習データを散布図としてプロット
ax.scatter(x1, x2, y, color='blue', s=50, label='学習データ (実際の値)')

# 予測値を散布図としてプロット
ax.scatter(x1, x2, y_pred, color='red', s=50, marker='x', label='予測値')

# 回帰平面をプロット
ax.plot_surface(X1_grid, X2_grid, Y_grid, alpha=0.3, cmap='viridis')

# 視点の設定 (elev: 上下の角度, azim: 左右の回転角度)
# この値を調整することで見え方が変わります
ax.view_init(elev=20, azim=45)

# ラベルとタイトルの設定
ax.set_xlabel('小テスト (x1)')
ax.set_ylabel('睡眠時間 (x2)')
ax.set_zlabel('期末テスト (y)')
ax.set_title('学習データ、予測値、および回帰平面の3Dプロット')
ax.legend()

plt.show()

マウスでドラッグして視点を変えられる可視化

In [ ]:
# データの準備
x1 = X_train_np[:, 0] # 小テスト
x2 = X_train_np[:, 1] # 睡眠時間
y = y_train_np        # 期末テスト

# 予測値の計算
y_pred = model.predict(X_train_np)

# 回帰平面を描画するためのグリッドデータを作成
x1_range = np.linspace(x1.min() - 0.5, x1.max() + 0.5, 20)
x2_range = np.linspace(x2.min() - 0.5, x2.max() + 0.5, 20)
X1_grid, X2_grid = np.meshgrid(x1_range, x2_range)

# 各グリッドポイントでの予測値を計算
Y_grid = model.intercept_ + model.coef_[0] * X1_grid + model.coef_[1] * X2_grid

# Plotly フィギュアの作成
fig = go.Figure()

# 1. 実際の学習データを散布図として追加
fig.add_trace(go.Scatter3d(
    x=x1,
    y=x2,
    z=y,
    mode='markers',
    marker=dict(size=6, color='blue', opacity=0.8),
    name='学習データ (実際の値)'
))

# 2. 予測値を散布図として追加
fig.add_trace(go.Scatter3d(
    x=x1,
    y=x2,
    z=y_pred,
    mode='markers',
    marker=dict(size=6, color='red', symbol='x', opacity=0.8),
    name='予測値'
))

# 3. 回帰平面を追加
fig.add_trace(go.Surface(
    x=X1_grid,
    y=X2_grid,
    z=Y_grid,
    opacity=0.5,
    colorscale='Viridis',
    showscale=False,
    name='回帰平面'
))

# レイアウトの設定
fig.update_layout(
    title='学習データ、予測値、および回帰平面の3Dインタラクティブプロット',
    scene=dict(
        xaxis_title='小テスト (x1)',
        yaxis_title='睡眠時間 (x2)',
        zaxis_title='期末テスト (y)'
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    width=800,
    height=600
)

fig.show()

## 課題1:その場で与えたデータ

チーム課題 teamL02 のデータを，下のように値をてコピーして入力して，偏回帰係数を求めるところまでやろう

In [ ]:
X2_train_np=np.array([[1,2],[3,4]])
y2_train_np=np.array([5,6])

In [ ]:
#

## 課題2:個人別課題データ

個人別に，MoodleのE02_02出題用からダウンロードしたCSVデータを，E00_04を参照して左側のフォルダ内にアップロードし，偏回帰係数を求めるところまでやろう．結果はこのNotebookの最初に書こう．

In [ ]:
df3=pd.read_csv("sample_data/california_housing_train.csv") # ファイル名を変更する必要

In [ ]:
# X3_train=
# y3_tran=